# CI/CD Smoke Tests and Fixtures
Companion notebook for the **CI/CD Smoke Tests and Fixtures** section of Chapter 7.

Smoke tests are a fast gate: imports work, config loads, startup succeeds, and one basic agent flow returns a valid result. This notebook installs the scaffold and **actually runs** its smoke tests with `pytest`, rather than just displaying the code.

`sys.executable` (not a bare `pip`/`pytest`) makes sure every command below runs against this notebook's own kernel environment, not whatever happens to be first on `PATH`.

In [1]:
import sys

!cd agent-app && "{sys.executable}" -m pip install --quiet -e ".[test]"

The fixtures, in `agent-app/tests/conftest.py` -- shared setup so each test starts from a known baseline:

In [2]:
print(open("agent-app/tests/conftest.py").read())

import pytest


@pytest.fixture
def test_config(monkeypatch):
    monkeypatch.setenv("APP_ENV", "test")
    monkeypatch.setenv("MODEL_NAME", "fake-model")
    return {"env": "test", "model": "fake-model"}


@pytest.fixture
def fake_tool_result():
    return {"status": "ok", "answer": "Order found"}


@pytest.fixture
def order_state():
    return {
        "user_id": "u-101",
        "intent": "check_order",
        "memory": {},
        "observations": [],
    }


@pytest.fixture
def fake_order_tool():
    return {
        "order_id": "123",
        "status": "shipped",
        "eta": "2026-04-30",
    }



The smoke tests themselves, in `agent-app/tests/test_smoke.py`:

In [3]:
print(open("agent-app/tests/test_smoke.py").read())

from support_agent.config import load_config
from support_agent.app import create_agent


def test_config_loads(test_config):
    cfg = load_config()
    assert cfg.env == "test"


def test_basic_agent_flow(test_config, fake_tool_result):
    agent = create_agent(tool_result=fake_tool_result)
    result = agent.run("Check order 123")
    assert result["status"] == "ok"
    assert "answer" in result



In [4]:
!cd agent-app && "{sys.executable}" -m pytest tests/test_smoke.py -v

============================= test session starts =============================
platform win32 -- Python 3.13.5, pytest-8.3.2, pluggy-1.6.0 -- C:\Users\himan\AppData\Local\Temp\claude\C--Users-himan\eec66fed-60c1-4f01-b9be-916a12ead6c3\scratchpad\nb_build\testenv7\Scripts\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\himan\AppData\Local\Temp\claude\C--Users-himan\eec66fed-60c1-4f01-b9be-916a12ead6c3\scratchpad\nb_build\output7\agent-app
configfile: pyproject.toml
plugins: asyncio-0.24.0
asyncio: mode=Mode.STRICT, default_loop_scope=None
collecting ... collected 2 items

tests/test_smoke.py::test_config_loads PASSED                            [ 50%]
tests/test_smoke.py::test_basic_agent_flow PASSED                        [100%]

============================== 2 passed in 0.03s ==============================


C:\Users\himan\AppData\Local\Temp\claude\C--Users-himan\eec66fed-60c1-4f01-b9be-916a12ead6c3\scratchpad\nb_build\testenv7\Lib\site-packages\pytest_asyncio\plugin.py:208: PytestDeprecationWarning: The configuration option "asyncio_default_fixture_loop_scope" is unset.
The event loop scope for asynchronous fixtures will default to the fixture caching scope. Future versions of pytest-asyncio will default the loop scope for asynchronous fixtures to function scope. Set the default fixture loop scope explicitly in order to avoid unexpected behavior in the future. Valid fixture loop scopes are: "function", "class", "module", "package", "session"

  warnings.warn(PytestDeprecationWarning(_DEFAULT_FIXTURE_LOOP_SCOPE_UNSET))


A good smoke-test failure should tell you what broke without digging through long logs. Keep this suite small, name tests by behavior, and avoid real network calls -- if you need deeper prompt or model-quality evaluation, run that in a separate, slower stage.